In [1]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer

from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt

In [2]:
# import csv file as dataframe
df = pd.read_csv("sentiment_tweets_2022.csv")


/var/folders/1n/4n926d913_94f_hb0lp693200000gn/T/ipykernel_13725/1963080322.py:2: DtypeWarning: Columns (10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("sentiment_tweets_2022.csv")


In [3]:
# copy dataframe into new variable to prevent changes to original dataframe
tweets = df.copy()

In [4]:
import ssl

ssl._create_default_https_context = ssl._create_unverified_context
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to
[nltk_data]     /Users/aashishtangnami/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aashishtangnami/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/aashishtangnami/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [5]:
#checking the len of features and their values of the data
tweets.shape

(451332, 27)

In [6]:
tweets.columns

Index(['Unnamed: 0', 'User', 'User_Location', 'Followers_Count',
       'User_Created', 'User_Description', 'User_Statuses_Count', 'Tweet_Date',
       'Num_Likes', 'Num_Retweets', 'Num_Quotes', 'Num_Replies', 'Url',
       'Coordinates', 'Tweet_Place', 'Hashtags', 'Tweet_Lang', 'Tweet_Source',
       'Tweet', 'Sentiment', 'Cleaned_Tweet', 'Pos_Tags', 'Adjectives',
       'Nouns', 'Verbs', 'Named Entities', 'All Entities'],
      dtype='object')

In [7]:
# checking if there are any null values.
tweets.isna().sum()

Unnamed: 0                  0
User                        0
User_Location          162833
Followers_Count             0
User_Created                0
User_Description        69280
User_Statuses_Count         0
Tweet_Date                  0
Num_Likes                   1
Num_Retweets                1
Num_Quotes                  0
Num_Replies                 0
Url                         0
Coordinates            446109
Tweet_Place            446112
Hashtags               310631
Tweet_Lang                  1
Tweet_Source                1
Tweet                       1
Sentiment                   0
Cleaned_Tweet              17
Pos_Tags                    0
Adjectives                  0
Nouns                       0
Verbs                       0
Named Entities              0
All Entities                0
dtype: int64

In [8]:
tweets.shape

(451332, 27)

In [9]:
tweets['Tweet'].head(5)

0    @_angelica_toy Happy Anniversary!!!....The Day...
1    @McfarlaneGlenda Happy Anniversary!!!....The D...
2    @thevivafrei @JustinTrudeau Happy Anniversary!...
3    @NChartierET Happy Anniversary!!!....The Day t...
4    @tabithapeters05 Happy Anniversary!!!....The D...
Name: Tweet, dtype: object

In [10]:
tweets['Sentiment'].head(5)

0    0.2444
1    0.2444
2    0.2444
3    0.2444
4    0.2444
Name: Sentiment, dtype: float64

In [11]:
tweets['Cleaned_Tweet'].head(5)

0    _angelica_toy Happy Anniversary!!!    The Day ...
1    McfarlaneGlenda Happy Anniversary!!!    The Da...
2    thevivafrei JustinTrudeau Happy Anniversary!!!...
3    NChartierET Happy Anniversary!!!    The Day th...
4    tabithapeters05 Happy Anniversary!!!    The Da...
Name: Cleaned_Tweet, dtype: object

In [12]:
tweets['Cleaned_Tweet'].shape
#dropping the null values
tweets['Cleaned_Tweet'].isna().sum()
tweets.dropna(subset=['Cleaned_Tweet'], inplace=True)
print(tweets['Cleaned_Tweet'].isna().sum())

0


Text Pre-Processing

In [13]:
def clean_tweets(tweets):
    """ basic preprocessing of tweets"""
    tweets = re.sub(r'<.*?>', '', tweets)  # This regex removes any HTML tags
    tweets = re.sub(r'[^a-zA-Z0-9\s]', '', tweets)  # This regex removes any special characters
    tweets = re.sub(r'[^\w\s]', '', tweets)  # This regex removes any punctuation
    tweets = tweets.lower()  # convert text to lower-case
    return tweets
    

In [14]:
# This line calls the function above and applies it to the 'Tweet' column
cleaned_tweets = tweets['Cleaned_Tweet'].apply(lambda tweet: clean_tweets(tweet))


In [15]:
cleaned_tweets.shape

(451315,)

In [16]:
def tweets_tokenize(tweets):
    return tweets.apply(lambda x: word_tokenize(x))


In [17]:
tokenized_tweets = tweets_tokenize(cleaned_tweets)

In [18]:
tokenized_tweets.shape

(451315,)

In [19]:


# define a function to remove stopwords from a list of tokens
def remove_stopwords(tokens):
    stop_words = set(stopwords.words('english'))
    # filter out any stop words from the list of tokens
    filtered_tokens = [token for token in tokens if not token.lower() in stop_words]
    
    return filtered_tokens

In [20]:


# define a function to perform lemmatization on a list of tokens
def lemmatize_tokens(tokens):
    # initialize the lemmatizer
    lemmatizer = WordNetLemmatizer()
    # remove any None values from the list of tokens
    tokens = [t for t in tokens if t is not None]
    
    # lemmatize each token in the list and store the result in a new list
    lemmatized_tokens = []
    for token in tokens:
        if isinstance(token, str):
            lemmatized_tokens.append(lemmatizer.lemmatize(token))
        elif isinstance(token, list):
            lemmatized_subtokens = lemmatize_tokens(token)
            lemmatized_tokens.extend(lemmatized_subtokens)

    return lemmatized_tokens

In [21]:
tokenized_tweets_no_stopwords = tokenized_tweets.apply(lambda x: remove_stopwords(x))

In [22]:
tokenized_tweets_no_stopwords[1:10]

1    [mcfarlaneglenda, happy, anniversary, day, fre...
2    [thevivafrei, justintrudeau, happy, anniversar...
3    [nchartieret, happy, anniversary, day, freedum...
4    [tabithapeters05, happy, anniversary, day, fre...
5    [justicestyle, happy, anniversary, day, freedu...
6    [praiset22112963, emergenciesact, ikwilson, ha...
7    [parnel1123, realandyleeshow, happy, anniversa...
8                     [freedom, convoy, inkblot, test]
9    [wsonlinenews, davidkrayden, happy, anniversar...
Name: Cleaned_Tweet, dtype: object

In [23]:
lemmatized_tokens = tokenized_tweets_no_stopwords.apply(lambda x: lemmatize_tokens(x))


In [24]:
lemmatized_tokens[1:10]

1    [mcfarlaneglenda, happy, anniversary, day, fre...
2    [thevivafrei, justintrudeau, happy, anniversar...
3    [nchartieret, happy, anniversary, day, freedum...
4    [tabithapeters05, happy, anniversary, day, fre...
5    [justicestyle, happy, anniversary, day, freedu...
6    [praiset22112963, emergenciesact, ikwilson, ha...
7    [parnel1123, realandyleeshow, happy, anniversa...
8                     [freedom, convoy, inkblot, test]
9    [wsonlinenews, davidkrayden, happy, anniversar...
Name: Cleaned_Tweet, dtype: object

In [25]:
value_range = df['Sentiment'].describe().loc[['min', 'max']]
value_range



min   -0.9927
max    0.9976
Name: Sentiment, dtype: float64

In [26]:
tweets['Cleaned_Tweet'].isna().sum()

0

Checking outliers

In [27]:

Q1 = tweets['Sentiment'].quantile(0.25)
Q3 = tweets['Sentiment'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR
lower_bound = Q1 - 1.5 * IQR
outliers = tweets[(tweets['Sentiment'] < lower_bound) | (tweets['Sentiment'] > upper_bound)]

In [28]:
outliers.shape

(0, 27)

Changing sentiments into class labels

In [29]:
# Convert continuous sentiment scores into discrete class labels
threshold = 0.2
labels = []
for score in tweets['Sentiment']:
    if score > threshold:
        label = 'positive'
    elif score < -threshold:
        label = 'negative'
    else:
        label = 'neutral'
    labels.append(label)
class_labels = np.array(labels)

In [30]:
class_labels

array(['positive', 'positive', 'positive', ..., 'positive', 'positive',
       'positive'], dtype='<U8')

Train Test Data Split.

In [31]:
# Split the data into training and testing sets
X_train_raw, X_test_raw, y_train, y_test = train_test_split(lemmatized_tokens, class_labels, test_size=0.4, random_state=42)


In [32]:
# Convert the lemmatized tokens into feature vectors using TF-IDF
vectorizer = TfidfVectorizer(ngram_range=(1, 1))
X_train_tfidf = vectorizer.fit_transform([' '.join(tokens) for tokens in X_train_raw])
X_test_tfidf = vectorizer.transform([' '.join(tokens) for tokens in X_test_raw])


Using Naive Bayes classifier 

In [33]:
# Use chi-squared test to select top 40000 features
fs = SelectKBest(score_func=chi2, k=40000)
X_train_fs = fs.fit_transform(X_train_tfidf, y_train)
X_test_fs = fs.transform(X_test_tfidf)

# Apply SMOTE oversampling to the training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_fs, y_train)

# Train a Naive Bayes classifier on the resampled training set with hyperparameter tuning
nb_clf = MultinomialNB()
param_grid = {'alpha': [0.1, 1.0, 10.0]}
clf = GridSearchCV(nb_clf, param_grid, cv=5)
clf.fit(X_train_resampled, y_train_resampled)

y_train_pred = clf.predict(X_train_fs)
train_accuracy = accuracy_score(y_train, y_train_pred)
print("Training accuracy:", train_accuracy)

# Use the trained model to predict the sentiment of the testing set and evaluate its performance
y_test_pred = clf.predict(X_test_fs)
test_accuracy = accuracy_score(y_test, y_test_pred)
print("Testing accuracy:", test_accuracy)

# Generate a confusion matrix for the testing set
cm = confusion_matrix(y_test, y_test_pred)
print("Confusion matrix:\n", cm)




Training accuracy: 0.731591755942819
Testing accuracy: 0.6682804692952816
Confusion matrix:
 [[21030  4012  5402]
 [ 8521 14499  7736]
 [21917 12296 85113]]


Observations from the confusion matrix:

- The confusion matrix has 3 rows and 3 columns, corresponding to the 3 classes in the dataset.
- The diagonal elements represent the number of correct predictions (true positive or TP) for each class.
- The off-diagonal elements represent incorrect predictions.

Specific observations from this confusion matrix:

- The model correctly predicted 21030 instances of the first class, 14499 instances of the second class, and 85113 instances of the third class.
- However, it incorrectly predicted 8521 instances of the first class, 4012 instances of the second class, and 12296 instances of the third class.

Observations from the accuracy scores:

- Training accuracy is the proportion of correct predictions made by the model on the training set.
- Testing accuracy is the proportion of correct predictions made on the testing set.
- In this case, the model has a higher accuracy on the training set (73.16%) compared to the testing set (66.83%).

This suggests possible overfitting of the model on the training data.

Bidirectional LSTM 

In [ ]:
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Embedding, LSTM, Dense
from keras.layers import Dropout, Bidirectional
from keras.callbacks import EarlyStopping
from keras.utils import to_categorical

# Prepare your data
X_train, X_test, y_train, y_test = train_test_split(lemmatized_tokens, class_labels, test_size=0.2, random_state=42)
max_words = 5000 # Set the maximum number of words in your vocabulary
max_len = 100 # Set the maximum length of each sequence

# Vectorize your text data
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)

X_test_seq = tokenizer.texts_to_sequences(X_test)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

# Convert string labels to numerical labels using the mapping
label_to_num = {'positive': 2, 'neutral': 1, 'negative': 0}
y_train_numerical = np.array([label_to_num[label] for label in y_train])
y_test_numerical = np.array([label_to_num[label] for label in y_test])
num_classes = 3

# Convert numerical labels to categorical format
y_train_categorical = to_categorical(y_train_numerical, num_classes=num_classes)
y_test_categorical = to_categorical(y_test_numerical, num_classes=num_classes)

embedding_dim = 128
lstm_units = 64
dropout_rate = 0.5
num_classes = 3

#model architecture
model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len))
model.add(Bidirectional(LSTM(units=lstm_units, dropout=dropout_rate, recurrent_dropout=dropout_rate)))
model.add(Dropout(dropout_rate))
model.add(Dense(units=num_classes, activation='softmax'))

# Compile the model 
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train and validate the model
history = model.fit(X_train_pad, y_train_categorical, epochs=10, batch_size=32,
                    validation_split=0.2, callbacks=[EarlyStopping(patience=2)])

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test_pad, y_test_categorical, verbose=0)
print(f"Testing accuracy: {test_accuracy}, Testing loss: {test_loss}")

In [ ]:
# Make predictions on new text data
new_texts = ["This is a positive sentence.", "This is a negative sentence."]
new_texts_seq = tokenizer.texts_to_sequences(new_texts)
new_texts_pad = pad_sequences(new_texts_seq, maxlen=max_len)
predictions = model.predict(new_texts_pad)

# Print the predicted labels
for i in range(len(predictions)):
    if predictions[i] > 0.5:
        print(f"'{new_texts[i]}' is predicted to be positive.")
    else:
        print(f"'{new_texts[i]}' is predicted to be negative.")

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test_pad, y_test, verbose=0)
print(f"Testing accuracy: {test_accuracy}, Testing loss: {test_loss}")

In [ ]:


# Plot the training and validation loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Val'], loc='upper right')
plt.show()

# Plot the training and validation accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Val'], loc='lower right')
plt.show()
